# Marine heatwave detection: from daily observations to event records

This lab focuses on the event-detection stage. It uses a synthetic daily temperature series and precomputed threshold/climatology arrays. The threshold and climatology here are intentionally simple teaching inputs, not a scientifically estimated seasonal baseline.

The widely used Hobday framework uses a seasonally varying 90th-percentile threshold, a minimum duration of five consecutive days and a short-gap joining rule. This notebook demonstrates run detection and basic event metrics; it does not implement the full climatology calculation or gridded spatial tracking.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'implementations/python/marine-heatwave-detection').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from a repository checkout.')
sys.path.insert(0, str(root / 'implementations/python/marine-heatwave-detection'))
from marine_heatwave_detection import detect_events


## 1. Build a synthetic daily series

We combine a seasonal cycle, background noise and three warm episodes. The threshold and climatological mean are known by construction.

In [ ]:
rng = np.random.default_rng(2026)
days = np.arange(180)
seasonal_mean = 18 + 2 * np.sin(2 * np.pi * days / 365)
temperature = seasonal_mean + rng.normal(0, 0.45, days.size)
temperature[30:37] += 2.0
temperature[39:45] += 2.2  # two-day gap: should join after qualifying runs
temperature[100:104] += 2.4  # four days: below the five-day minimum
temperature[140:148] += 2.8
threshold = seasonal_mean + 1.0
climatology = seasonal_mean
events = detect_events(temperature, threshold, climatology)
[(e.start_index, e.end_index, e.duration_days, e.exceedance_days) for e in events]

## 2. Visualise the event intervals

The shaded intervals show detected event spans. The threshold is used to identify exceedance; intensity metrics are calculated relative to the climatological mean.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(days, temperature, label='Daily temperature', linewidth=1.2)
ax.plot(days, threshold, label='Illustrative threshold', linestyle='--')
ax.plot(days, climatology, label='Illustrative climatology', alpha=0.8)
for i, event in enumerate(events):
    ax.axvspan(event.start_index, event.end_index, alpha=0.18,
               label='Detected event' if i == 0 else None)
ax.set_xlabel('Day index (assumed consecutive daily observations)')
ax.set_ylabel('Temperature (synthetic units)')
ax.set_title('Point-series marine heatwave event detection')
ax.legend(ncol=2)
plt.show()

## 3. Inspect event metrics

Duration includes any joined gap days. Exceedance days count only observations above the threshold. The cumulative intensity is the sum of temperature anomalies relative to the climatological mean over the event interval.

In [ ]:
for number, event in enumerate(events, start=1):
    print(f'Event {number}: days {event.start_index}–{event.end_index}')
    print(f'  duration: {event.duration_days} days')
    print(f'  threshold-exceedance days: {event.exceedance_days}')
    print(f'  mean intensity: {event.mean_intensity:.2f}')
    print(f'  maximum intensity: {event.maximum_intensity:.2f}')
    print(f'  cumulative intensity: {event.cumulative_intensity:.2f} degree-days')


## 4. Missing observations break the sequence

Missing values cannot be assumed to be cool days. The implementation treats a missing value as a hard break and will not join events across it.

In [ ]:
with_gap = temperature.copy()
with_gap[37] = np.nan
events_with_gap = detect_events(with_gap, threshold, climatology)
print('Events without missing value:', len(events))
print('Events with missing value inside first warm episode:', len(events_with_gap))


## 5. From event detection to spatial tracking

This implementation analyses one location. A gridded workflow would detect events independently at each cell, label connected regions on each day and then link those regions through time. Tracking requires explicit rules for adjacency, splits, merges, missing data, coastlines and periodic longitude.

## Investigate

1. Change the minimum duration to 3 and 7 days. Which events appear or disappear?
2. Change the maximum gap from 2 to 0. Which event splits?
3. Insert a missing value into the gap between two qualifying runs. Does the event merge?
4. Replace the fixed threshold with a varying threshold array. Why is this closer to a real seasonal method?
5. Explain why a connected-component label on one day is not a trajectory.

Continue with the [method lesson](../methods/marine-heatwave-detection.md), [implementation walkthrough](../implementations/python/marine-heatwave-detection/implementation-walkthrough.md) and [exercise](../exercises/marine-heatwave-detection.md).